# Space Cadet: re-entry classification

Predict whether a known re-entered object has 365 days or less remaining at an observation. Baseline models and four candidates share a frozen 70/15/15 allocation **by object**, with all observations of each object kept together. Observation percentages may differ slightly. Objects are stratified by their proportion of positive observations, using a fixed seed chosen before fitting.

This replaces the small fixed training sample. All eligible known re-entered objects are used, including previously evaluated objects. The reshuffled test cohort is a **development holdout, not an untouched final test**. Earlier results are archived under `data/processed/model_archive_before_701515`; old split manifests remain as provenance.

This experiment measures generalization to different objects in this panel, not future dates or all objects still in orbit. The existing lifetime-based observation sampling and known-re-entry selection remain limitations; changing splits does not fix those issues. Model settings and threshold 0.5 are fixed before evaluation. No hyperparameter search is performed here.


## Shared data, split checks, and evaluation helpers


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, average_precision_score,
                             roc_auc_score, precision_score, recall_score,
                             brier_score_loss, confusion_matrix)
from threadpoolctl import threadpool_limits

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src/Models/classification_split.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.Models.classification_split import load_cohorts, TARGET, SEED

data, train_mask, valid_mask, test_mask, solar_features = load_cohorts(PROJECT_ROOT)
feature_groups = json.loads((PROJECT_ROOT / "data/processed/feature_groups.json").read_text())
no_sun_features = feature_groups["ablation"]["baseline_no_sun"]
weather = set(feature_groups["features"]["space_weather_trailing"])
assert not set(no_sun_features) & weather
assert set(solar_features) - set(no_sun_features) == weather
split_summary = []
for name, mask in [("Training", train_mask), ("Validation", valid_mask), ("Test", test_mask)]:
    subset = data.loc[mask]
    split_summary.append({"Split": name, "Observations": len(subset),
                          "Objects": subset.norad_cat_id.nunique(),
                          "Observation share": len(subset)/len(data),
                          "Object share": subset.norad_cat_id.nunique()/data.norad_cat_id.nunique(),
                          "Positive rate": subset[TARGET].mean()})
print(pd.DataFrame(split_summary).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Verified: all eligible observations assigned, zero object overlap, both classes in every split.")
print("Test status: reused-panel development holdout, not an untouched final evaluation.")
y_train = data.loc[train_mask, TARGET]
groups_train = data.loc[train_mask, "norad_cat_id"]
cv_folds = list(GroupKFold(n_splits=5).split(data.loc[train_mask], y_train, groups_train))
for fit_rows, score_rows in cv_folds:
    assert not set(groups_train.iloc[fit_rows]) & set(groups_train.iloc[score_rows])
    assert y_train.iloc[fit_rows].nunique() == y_train.iloc[score_rows].nunique() == 2
DECISION_THRESHOLD = 0.5
RUN_TEST_EVALUATION = False  # Explicit runner --include-test enables this for that run only.
training_results, evaluation_results = {}, {}


     Split  Observations  Objects  Observation share  Object share  Positive rate
  Training        273309    24141             0.7007        0.7000         0.4606
Validation         58407     5173             0.1498        0.1500         0.4587
      Test         58308     5174             0.1495        0.1500         0.4588
Verified: all eligible observations assigned, zero object overlap, both classes in every split.
Test status: reused-panel development holdout, not an untouched final evaluation.


In [2]:
def make_model(columns, algorithm):
    numeric = data.loc[train_mask, columns].select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical = [column for column in columns if column not in numeric]
    numeric_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if algorithm == "logistic":
        numeric_steps.append(("scale", StandardScaler()))
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=algorithm == "logistic")),
        ]), categorical),
    ], sparse_threshold=0.3 if algorithm == "logistic" else 0)
    if algorithm == "logistic":
        estimator = LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=SEED)
    else:
        estimator = HistGradientBoostingClassifier(
            learning_rate=0.1, max_iter=100, max_leaf_nodes=31, min_samples_leaf=20,
            l2_regularization=0.0, class_weight="balanced", early_stopping=False,
            random_state=SEED,
        )
    return Pipeline([("preprocess", preprocessor), ("classifier", estimator)])


def fit_and_report(columns, algorithm, name):
    model = make_model(columns, algorithm)
    X_train = data.loc[train_mask, columns]
    with threadpool_limits(limits=4):
        scores = cross_validate(model, X_train, y_train, cv=cv_folds,
            scoring={"average_precision": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},
            n_jobs=1, error_score="raise")
        model.fit(X_train, y_train)
    print(f"{name}: {len(columns)} inputs; {len(X_train):,} training observations; {groups_train.nunique():,} objects")
    training_results[name] = {}
    for metric in ["average_precision", "roc_auc", "brier"]:
        values = scores["test_" + metric] * (-1 if metric == "brier" else 1)
        training_results[name][metric] = {"mean": float(values.mean()), "std": float(values.std())}
        print(f"Grouped training CV {metric}: {values.mean():.4f} +/- {values.std():.4f}")
    return model


def evaluate(model, columns, mask, name, cohort):
    labels = data.loc[mask, TARGET]
    with threadpool_limits(limits=4):
        probability = model.predict_proba(data.loc[mask, columns])[:, 1]
    prediction = (probability >= DECISION_THRESHOLD).astype("int8")
    metrics = {
        "Accuracy": accuracy_score(labels, prediction),
        "Balanced accuracy": balanced_accuracy_score(labels, prediction),
        "Average precision": average_precision_score(labels, probability),
        "ROC AUC": roc_auc_score(labels, probability),
        "Precision": precision_score(labels, prediction, zero_division=0),
        "Recall": recall_score(labels, prediction, zero_division=0),
        "Brier score": brier_score_loss(labels, probability),
        "Training-majority baseline accuracy": accuracy_score(labels, np.full(len(labels), int(y_train.mean() >= 0.5))),
        "Training-rate baseline Brier score": brier_score_loss(labels, np.full(len(labels), y_train.mean())),
    }
    evaluation_results[(name, cohort)] = metrics
    print(f"{name} - {cohort}, threshold {DECISION_THRESHOLD:.2f}")
    print(f"Observations: {len(labels):,}; positive rate: {labels.mean():.4f}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print("Confusion matrix (rows actual 0/1; columns predicted 0/1):")
    print(confusion_matrix(labels, prediction, labels=[0, 1]))


## 1. Baseline models

These are the standard reference points for a binary classifier. The majority-class baseline always predicts the most common training class. The training-rate baseline always outputs the training positive rate as a probability and converts it to a class at threshold 0.5. Both are fitted from the training cohort only and use no features. Their purpose is to show the minimum performance a candidate model must beat.

### Training, validation, and test performance

The same metrics and cohorts are used for all splits. The test cohort is the reused-panel development holdout described above, not an untouched final test. Constant-probability baselines have no meaningful ROC ranking, so ROC AUC is reported as undefined; Brier score evaluates their probability quality.


In [3]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             average_precision_score, precision_score, recall_score,
                             brier_score_loss, confusion_matrix)

majority_class = int(y_train.mean() >= 0.5)
training_rate = float(y_train.mean())
baseline_results = {}

def evaluate_baselines(mask, cohort):
    labels = data.loc[mask, TARGET]
    majority_probability = np.full(len(labels), majority_class, dtype=float)
    rate_probability = np.full(len(labels), training_rate, dtype=float)
    rows = []
    for name, probability in [
        ("Majority-class baseline", majority_probability),
        ("Training-rate probability baseline", rate_probability),
    ]:
        prediction = (probability >= DECISION_THRESHOLD).astype("int8")
        metrics = {
            "Accuracy": accuracy_score(labels, prediction),
            "Balanced accuracy": balanced_accuracy_score(labels, prediction),
            "Average precision": average_precision_score(labels, probability),
            "ROC AUC": np.nan,
            "Precision": precision_score(labels, prediction, zero_division=0),
            "Recall": recall_score(labels, prediction, zero_division=0),
            "Brier score": brier_score_loss(labels, probability),
        }
        baseline_results[(name, cohort)] = metrics
        rows.append({"Model": name, "Cohort": cohort, **metrics})
        print(f"{name} - {cohort}, threshold {DECISION_THRESHOLD:.2f}")
        for metric, value in metrics.items():
            print(f"{metric}: {'undefined' if pd.isna(value) else f'{value:.4f}'}")
        print("Confusion matrix (rows actual 0/1; columns predicted 0/1):")
        print(confusion_matrix(labels, prediction, labels=[0, 1]))
    return rows

baseline_table = []
baseline_table.extend(evaluate_baselines(train_mask, "Training"))
baseline_table.extend(evaluate_baselines(valid_mask, "Validation"))
if RUN_TEST_EVALUATION:
    baseline_table.extend(evaluate_baselines(test_mask, "Test"))
else:
    print("Baseline test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")
baseline_table = pd.DataFrame(baseline_table)


Majority-class baseline - Training, threshold 0.50
Accuracy: 0.5394
Balanced accuracy: 0.5000
Average precision: 0.4606
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.4606
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[147418      0]
 [125891      0]]
Training-rate probability baseline - Training, threshold 0.50
Accuracy: 0.5394
Balanced accuracy: 0.5000
Average precision: 0.4606
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.2484
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[147418      0]
 [125891      0]]
Majority-class baseline - Validation, threshold 0.50
Accuracy: 0.5413
Balanced accuracy: 0.5000
Average precision: 0.4587
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.4587
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[31613     0]
 [26794     0]]
Training-rate probability baseline - Validation, threshold 0.50
Accuracy: 0.5413
Balanced accuracy: 0.5000
Average precision: 0.4587


## 1. Logistic regression without solar information


### Model description

Class-balanced logistic regression with L2 regularization, C=1, lbfgs solver, and max_iter=1000. Numeric inputs receive median imputation, missing indicators, and standardization. Categorical inputs receive missing-category imputation and one-hot encoding.

Uses 20 inputs: object, age, and orbit features; all solar/geomagnetic inputs excluded. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [4]:
model_1 = fit_and_report(no_sun_features, "logistic", "Logistic regression without solar information")


Logistic regression without solar information: 20 inputs; 273,309 training observations; 24,141 objects
Grouped training CV average_precision: 0.8989 +/- 0.0026
Grouped training CV roc_auc: 0.9029 +/- 0.0019
Grouped training CV brier: 0.1239 +/- 0.0011


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [5]:
evaluate(model_1, no_sun_features, valid_mask, "Logistic regression without solar information", "Validation")


Logistic regression without solar information - Validation, threshold 0.50
Observations: 58,407; positive rate: 0.4587
Accuracy: 0.8260
Balanced accuracy: 0.8232
Average precision: 0.8948
ROC AUC: 0.9002
Precision: 0.8249
Recall: 0.7881
Brier score: 0.1257
Training-majority baseline accuracy: 0.5413
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[27130  4483]
 [ 5677 21117]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [6]:
if RUN_TEST_EVALUATION:
    evaluate(model_1, no_sun_features, test_mask, "Logistic regression without solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Logistic regression without solar information - Test, threshold 0.50
Observations: 58,308; positive rate: 0.4588
Accuracy: 0.8274
Balanced accuracy: 0.8247
Average precision: 0.8980
ROC AUC: 0.9017
Precision: 0.8243
Recall: 0.7927
Brier score: 0.1243
Training-majority baseline accuracy: 0.5412
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[27034  4521]
 [ 5545 21208]]


## 2. Logistic regression with solar information


### Model description

Class-balanced logistic regression with L2 regularization, C=1, lbfgs solver, and max_iter=1000. Numeric inputs receive median imputation, missing indicators, and standardization. Categorical inputs receive missing-category imputation and one-hot encoding.

Uses 52 inputs: object, age, and orbit features plus 32 solar/geomagnetic inputs. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [7]:
model_2 = fit_and_report(solar_features, "logistic", "Logistic regression with solar information")


Logistic regression with solar information: 52 inputs; 273,309 training observations; 24,141 objects
Grouped training CV average_precision: 0.9043 +/- 0.0028
Grouped training CV roc_auc: 0.9077 +/- 0.0018
Grouped training CV brier: 0.1208 +/- 0.0012


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [8]:
evaluate(model_2, solar_features, valid_mask, "Logistic regression with solar information", "Validation")


Logistic regression with solar information - Validation, threshold 0.50
Observations: 58,407; positive rate: 0.4587
Accuracy: 0.8304
Balanced accuracy: 0.8282
Average precision: 0.9008
ROC AUC: 0.9054
Precision: 0.8241
Recall: 0.8014
Brier score: 0.1226
Training-majority baseline accuracy: 0.5413
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[27030  4583]
 [ 5321 21473]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [9]:
if RUN_TEST_EVALUATION:
    evaluate(model_2, solar_features, test_mask, "Logistic regression with solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Logistic regression with solar information - Test, threshold 0.50
Observations: 58,308; positive rate: 0.4588
Accuracy: 0.8333
Balanced accuracy: 0.8313
Average precision: 0.9032
ROC AUC: 0.9067
Precision: 0.8263
Recall: 0.8062
Brier score: 0.1215
Training-majority baseline accuracy: 0.5412
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[27021  4534]
 [ 5184 21569]]


## 3. Histogram gradient boosting without solar information


### Model description

Class-balanced histogram gradient boosting with 100 iterations, learning rate 0.1, 31 maximum leaves, minimum 20 samples per leaf, and L2 regularization 0. Numeric inputs receive median imputation and missing indicators; categorical inputs receive missing-category imputation and dense one-hot encoding. No scaling is needed. Automatic early stopping is disabled to avoid an internal row-level split that ignores object groups.

Uses 20 inputs: object, age, and orbit features; all solar/geomagnetic inputs excluded. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [10]:
model_3 = fit_and_report(no_sun_features, "boost", "Histogram gradient boosting without solar information")


Histogram gradient boosting without solar information: 20 inputs; 273,309 training observations; 24,141 objects
Grouped training CV average_precision: 0.9742 +/- 0.0009
Grouped training CV roc_auc: 0.9755 +/- 0.0005
Grouped training CV brier: 0.0604 +/- 0.0007


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [11]:
evaluate(model_3, no_sun_features, valid_mask, "Histogram gradient boosting without solar information", "Validation")


Histogram gradient boosting without solar information - Validation, threshold 0.50
Observations: 58,407; positive rate: 0.4587
Accuracy: 0.9169
Balanced accuracy: 0.9154
Average precision: 0.9743
ROC AUC: 0.9759
Precision: 0.9200
Recall: 0.8970
Brier score: 0.0601
Training-majority baseline accuracy: 0.5413
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[29522  2091]
 [ 2761 24033]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [12]:
if RUN_TEST_EVALUATION:
    evaluate(model_3, no_sun_features, test_mask, "Histogram gradient boosting without solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Histogram gradient boosting without solar information - Test, threshold 0.50
Observations: 58,308; positive rate: 0.4588
Accuracy: 0.9172
Balanced accuracy: 0.9160
Average precision: 0.9746
ROC AUC: 0.9759
Precision: 0.9173
Recall: 0.9009
Brier score: 0.0602
Training-majority baseline accuracy: 0.5412
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[29381  2174]
 [ 2652 24101]]


## 4. Histogram gradient boosting with solar information


### Model description

Class-balanced histogram gradient boosting with 100 iterations, learning rate 0.1, 31 maximum leaves, minimum 20 samples per leaf, and L2 regularization 0. Numeric inputs receive median imputation and missing indicators; categorical inputs receive missing-category imputation and dense one-hot encoding. No scaling is needed. Automatic early stopping is disabled to avoid an internal row-level split that ignores object groups.

Uses 52 inputs: object, age, and orbit features plus 32 solar/geomagnetic inputs. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [13]:
model_4 = fit_and_report(solar_features, "boost", "Histogram gradient boosting with solar information")


Histogram gradient boosting with solar information: 52 inputs; 273,309 training observations; 24,141 objects
Grouped training CV average_precision: 0.9786 +/- 0.0008
Grouped training CV roc_auc: 0.9799 +/- 0.0003
Grouped training CV brier: 0.0543 +/- 0.0006


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [14]:
evaluate(model_4, solar_features, valid_mask, "Histogram gradient boosting with solar information", "Validation")


Histogram gradient boosting with solar information - Validation, threshold 0.50
Observations: 58,407; positive rate: 0.4587
Accuracy: 0.9267
Balanced accuracy: 0.9258
Average precision: 0.9790
ROC AUC: 0.9806
Precision: 0.9244
Recall: 0.9151
Brier score: 0.0537
Training-majority baseline accuracy: 0.5413
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[29608  2005]
 [ 2276 24518]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [15]:
if RUN_TEST_EVALUATION:
    evaluate(model_4, solar_features, test_mask, "Histogram gradient boosting with solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Histogram gradient boosting with solar information - Test, threshold 0.50
Observations: 58,308; positive rate: 0.4588
Accuracy: 0.9249
Balanced accuracy: 0.9242
Average precision: 0.9789
ROC AUC: 0.9802
Precision: 0.9203
Recall: 0.9157
Brier score: 0.0544
Training-majority baseline accuracy: 0.5412
Training-rate baseline Brier score: 0.2483
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[29433  2122]
 [ 2255 24498]]


## Model comparison

All scores below come from the current 70/15/15 split. Test results are descriptive follow-up results; use training CV and validation for any later selection.


In [16]:
comparison = pd.DataFrame([
    {"Model": name, "Cohort": cohort, **metrics}
    for (name, cohort), metrics in evaluation_results.items()
])
print(comparison[["Model", "Cohort", "Accuracy", "Average precision", "ROC AUC", "Brier score"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


                                                Model     Cohort  Accuracy  Average precision  ROC AUC  Brier score
        Logistic regression without solar information Validation    0.8260             0.8948   0.9002       0.1257
        Logistic regression without solar information       Test    0.8274             0.8980   0.9017       0.1243
           Logistic regression with solar information Validation    0.8304             0.9008   0.9054       0.1226
           Logistic regression with solar information       Test    0.8333             0.9032   0.9067       0.1215
Histogram gradient boosting without solar information Validation    0.9169             0.9743   0.9759       0.0601
Histogram gradient boosting without solar information       Test    0.9172             0.9746   0.9759       0.0602
   Histogram gradient boosting with solar information Validation    0.9267             0.9790   0.9806       0.0537
   Histogram gradient boosting with solar information       Test    0.92